# RNNs & LSTMs: hands-on (optional)
**LLMs, Animated · Lesson P6.3**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; a ✓ means success.

You'll write one RNN step in NumPy, watch gradients vanish through time with PyTorch's autograd, implement gradient clipping, and race a vanilla RNN against an LSTM on a task that needs a long memory. Everything runs on a CPU in well under a minute.

Stuck? Solutions are at the very bottom.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(2)
print("torch", torch.__version__)

## 1 · One RNN step
$$h_t = \tanh(W h_{t-1} + U x_t + b)$$
Write `rnn_step(h, x, W, U, b)` in NumPy, then `rnn_run(xs, W, U, b)`, which starts from $h_0 = 0$ and returns the list of hidden states $[h_1, \dots, h_T]$.

In [ ]:
def rnn_step(h, x, W, U, b):
    # TODO
    raise NotImplementedError("Exercise 1")

def rnn_run(xs, W, U, b):
    # TODO: loop over the inputs, collecting every hidden state
    raise NotImplementedError("Exercise 1")

**Check 1.** The lesson's one-number example ($W = 0.5$, $U = 1$, inputs 1, 0, 0), then a vector version.

In [ ]:
hs = rnn_run([np.array([1.0]), np.array([0.0]), np.array([0.0])], np.array([[0.5]]), np.array([[1.0]]), np.zeros(1))
print([round(float(h[0]), 3) for h in hs])
assert np.allclose([h[0] for h in hs], [0.762, 0.363, 0.180], atol=1e-3)
rng = np.random.default_rng(0)
W, U, b = rng.normal(0, 0.3, (20, 20)), rng.normal(0, 0.3, (20, 10)), np.zeros(20)
hs = rnn_run(rng.normal(size=(7, 10)), W, U, b)
assert len(hs) == 7 and hs[-1].shape == (20,) and np.all(np.abs(hs[-1]) < 1)
print("parameters (input 10, hidden 20):", W.size + U.size + b.size)
assert W.size + U.size + b.size == 620
print("✓ exercise 1")

## 2 · Measure the vanishing gradient
How much does the **last** hidden state depend on the **first** input? Write `grad_to_first_input(cell, T)`:

1. make an input of shape `(1, T, 4)` with random values and `requires_grad=True`;
2. run it through `cell` (an `nn.RNN` or `nn.LSTM` with `batch_first=True`);
3. take the sum of the final time step's output, call `.backward()`;
4. return the norm of the gradient with respect to the **first** time step of the input, as a Python float.

In [ ]:
def grad_to_first_input(cell, T, seed=0):
    torch.manual_seed(seed)
    # TODO
    raise NotImplementedError("Exercise 2")

**Check 2.** For a freshly initialised vanilla RNN the gradient shrinks roughly exponentially with $T$. We plot it on a log scale.

In [ ]:
torch.manual_seed(0)
rnn = nn.RNN(4, 32, batch_first=True)
Ts = [1, 5, 10, 20, 40, 80]
g = [grad_to_first_input(rnn, T) for T in Ts]
for T, v in zip(Ts, g):
    print(f"T = {T:3d}: |dh_T / dx_1| = {v:.2e}")
assert g[0] > 1e-2
assert g[-1] < g[0] * 1e-6, "the gradient from 80 steps back should be tiny"
plt.semilogy(Ts, g, "o-")
plt.xlabel("T (steps between input and output)"); plt.ylabel("gradient norm (log)")
plt.title("vanilla RNN at initialisation"); plt.show()
print("✓ exercise 2")

## 3 · Gradient clipping
Write `clip_by_norm(grads, max_norm)`: given a list of tensors, compute their **global** norm $\sqrt{\sum \lVert g_i \rVert^2}$. If it exceeds `max_norm`, scale every tensor by `max_norm / norm`. Return the new list and the original norm.

In [ ]:
def clip_by_norm(grads, max_norm):
    # TODO
    raise NotImplementedError("Exercise 3")

**Check 3.** The lesson's example, then a comparison with `torch.nn.utils.clip_grad_norm_`.

In [ ]:
out, n = clip_by_norm([torch.tensor([30.0, 40.0])], 5.0)
assert abs(n - 50) < 1e-5 and torch.allclose(out[0], torch.tensor([3.0, 4.0]))
out, n = clip_by_norm([torch.tensor([0.3, 0.4])], 5.0)
assert torch.allclose(out[0], torch.tensor([0.3, 0.4])), "small gradients must be left alone"
torch.manual_seed(1)
lin = nn.Linear(10, 10)
lin(torch.randn(8, 10)).pow(2).sum().mul(100).backward()
mine, n = clip_by_norm([p.grad.clone() for p in lin.parameters()], 1.0)
ref = nn.utils.clip_grad_norm_(lin.parameters(), 1.0)
assert abs(n - ref.item()) < 1e-3 * max(1.0, n)
assert all(torch.allclose(a, p.grad, atol=1e-6) for a, p in zip(mine, lin.parameters()))
print(f"global norm was {n:.1f}; clipped to 1.0")
print("✓ exercise 3")

## 4 · Remember the first token
Each sequence is $T$ random tokens from a vocabulary of 4, one-hot encoded. The label is **the first token**. Everything after it is noise, so the network has to carry one piece of information across $T - 1$ steps.

Write `Model(kind, hidden)`: an `nn.RNN` or `nn.LSTM` (`batch_first=True`) followed by `nn.Linear(hidden, 4)` applied to the **last** time step's output. For the LSTM, set the **forget-gate bias** to 3 so the cell starts out remembering. PyTorch stores the gate biases in the order input, forget, cell, output, so the forget gate is the slice `[hidden:2*hidden]` of `bias_ih_l0` (zero the same slice of `bias_hh_l0`).

In [ ]:
K = 4

def make_batch(n, T, gen):
    seq = torch.randint(0, K, (n, T), generator=gen)
    return F.one_hot(seq, K).float(), seq[:, 0]

def train_model(model, T, steps=600, lr=3e-3, seed=0):
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for s in range(steps):
        x, y = make_batch(64, T, gen)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # clipping, as in exercise 3
        opt.step()
        losses.append(loss.item())
    x, y = make_batch(1000, T, torch.Generator().manual_seed(123))
    with torch.no_grad():
        acc = (model(x).argmax(1) == y).float().mean().item()
    return acc, losses

In [ ]:
class Model(nn.Module):
    def __init__(self, kind="rnn", hidden=64, forget_bias=3.0):
        super().__init__()
        # TODO: self.rnn, self.out, and the LSTM forget-gate bias
        raise NotImplementedError("Exercise 4")

    def forward(self, x):
        # TODO
        raise NotImplementedError("Exercise 4")

**Check 4.** Shapes and parameter counts (PyTorch keeps two bias vectors per gate, so an LSTM layer has $4(H(H + I) + 2H)$ parameters).

In [ ]:
torch.manual_seed(0)
m_rnn, m_lstm = Model("rnn", 64), Model("lstm", 64)
x, _ = make_batch(3, 10, torch.Generator().manual_seed(0))
assert m_rnn(x).shape == (3, 4) and m_lstm(x).shape == (3, 4)
n_rnn = sum(p.numel() for p in m_rnn.rnn.parameters())
n_lstm = sum(p.numel() for p in m_lstm.rnn.parameters())
assert n_rnn == 64 * (64 + 4) + 2 * 64 and n_lstm == 4 * n_rnn
assert torch.all(m_lstm.rnn.bias_ih_l0[64:128] == 3.0)
print(f"RNN layer: {n_rnn:,} params, LSTM layer: {n_lstm:,} params")
print("✓ exercise 4")

## 5 · The race
Train both on short sequences ($T = 5$) and long ones ($T = 40$). Each run takes a few seconds. Then fill in `verdict`: the two accuracies at $T = 40$. Chance level is 25%.

In [ ]:
results = {}
for T in [5, 40]:
    for kind in ["rnn", "lstm"]:
        torch.manual_seed(0)
        t0 = time.time()
        acc, losses = train_model(Model(kind, 64), T)
        results[(kind, T)] = (acc, losses)
        print(f"T = {T:2d}  {kind:4s}: accuracy {acc:.1%}  ({time.time() - t0:.1f}s)")

fig, ax = plt.subplots(figsize=(7, 3))
for (kind, T), (acc, losses) in results.items():
    if T == 40:
        ax.plot(np.convolve(losses, np.ones(20) / 20, mode="valid"), label=f"{kind}, T = 40")
ax.axhline(np.log(4), ls="--", c="gray", label="chance (ln 4)")
ax.set_xlabel("training step"); ax.set_ylabel("loss"); ax.legend(); plt.show()

In [ ]:
# TODO: read the two T = 40 accuracies out of `results`
rnn_acc_40 = None
lstm_acc_40 = None

**Check 5**

In [ ]:
assert results[("rnn", 5)][0] > 0.95, "a short memory is easy for both"
assert lstm_acc_40 > 0.95, "the LSTM should carry the first token across 40 steps"
assert rnn_acc_40 < 0.5, "the vanilla RNN should be stuck near chance"
print(f"T = 40: RNN {rnn_acc_40:.0%} vs LSTM {lstm_acc_40:.0%}")
print("✓ exercise 5")

## Experiment
1. Set `forget_bias=0.0` (PyTorch's default) for the LSTM and train at $T = 40$ again. Does it still learn within 600 steps? Why might the bias matter?
2. Run `grad_to_first_input` on a fresh `nn.LSTM(4, 32, batch_first=True)` with its forget bias set to 3, and compare with the RNN curve from exercise 2.
3. Try `nn.GRU`. It has three weight blocks instead of the LSTM's four: check its parameter count.
4. Push $T$ to 80 or 120. Where does the LSTM start to struggle?

In [ ]:
torch.manual_seed(0)
lstm = nn.LSTM(4, 32, batch_first=True)
with torch.no_grad():
    lstm.bias_ih_l0[32:64].fill_(3.0); lstm.bias_hh_l0[32:64].fill_(0.0)
g_lstm = [grad_to_first_input(lstm, T) for T in Ts]
plt.semilogy(Ts, g, "o-", label="vanilla RNN")
plt.semilogy(Ts, g_lstm, "o-", label="LSTM (forget bias 3)")
plt.xlabel("T"); plt.ylabel("gradient norm (log)"); plt.legend(); plt.show()

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def rnn_step(h, x, W, U, b):
    return np.tanh(W @ h + U @ x + b)

def rnn_run(xs, W, U, b):
    h = np.zeros(W.shape[0])
    hs = []
    for x in xs:
        h = rnn_step(h, x, W, U, b)
        hs.append(h)
    return hs

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def grad_to_first_input(cell, T, seed=0):
    torch.manual_seed(seed)
    x = torch.randn(1, T, 4, requires_grad=True)
    out, _ = cell(x)
    out[:, -1].sum().backward()
    return x.grad[0, 0].norm().item()

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def clip_by_norm(grads, max_norm):
    norm = torch.sqrt(sum((g ** 2).sum() for g in grads))
    scale = max_norm / norm if norm > max_norm else 1.0
    return [g * scale for g in grads], norm.item()

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
class Model(nn.Module):
    def __init__(self, kind="rnn", hidden=64, forget_bias=3.0):
        super().__init__()
        cls = nn.LSTM if kind == "lstm" else nn.RNN
        self.rnn = cls(K, hidden, batch_first=True)
        self.out = nn.Linear(hidden, K)
        if kind == "lstm":
            with torch.no_grad():
                self.rnn.bias_ih_l0[hidden:2 * hidden].fill_(forget_bias)
                self.rnn.bias_hh_l0[hidden:2 * hidden].fill_(0.0)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.out(out[:, -1])

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
rnn_acc_40 = results[("rnn", 40)][0]
lstm_acc_40 = results[("lstm", 40)][0]